# Adult Census Preprocessing

This notebook applies the preprocessing decisions identified during EDA and prepares the data for modeling.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATA_PATH = "adult.csv"
TARGET = "income"

## Load Data

In [ ]:
df = pd.read_csv(DATA_PATH, na_values="?")

# Use Python-friendly column names.
df = df.rename(columns=lambda col: col.strip().replace(".", "_"))

print(f"Raw shape: {df.shape}")
df.head()

## Remove Duplicate Rows

In [ ]:
df_no_exact_duplicates = df.drop_duplicates()

print(f"Rows before removing exact duplicates: {len(df)}")
print(f"Exact duplicate rows removed: {len(df) - len(df_no_exact_duplicates)}")
print(f"Rows after removing exact duplicates: {len(df_no_exact_duplicates)}")

In [ ]:
def values_differ(value_a, value_b):
    both_missing = pd.isna(value_a) and pd.isna(value_b)
    return not (both_missing or value_a == value_b)


exact_duplicate_rows = df[df.duplicated(keep=False)]
near_duplicate_indices = set()

for i in range(len(exact_duplicate_rows)):
    for j in range(i + 1, len(exact_duplicate_rows)):
        row_i = exact_duplicate_rows.iloc[i]
        row_j = exact_duplicate_rows.iloc[j]
        diff_cols = [
            col for col in df.columns if values_differ(row_i[col], row_j[col])
        ]

        if set(diff_cols) == {"age", "fnlwgt"}:
            near_duplicate_indices.update([exact_duplicate_rows.index[i], exact_duplicate_rows.index[j]])

near_duplicate_indices = sorted(near_duplicate_indices)

print(f"Rows involved in near-duplicate groups: {len(near_duplicate_indices)}")
df.loc[near_duplicate_indices]

In [ ]:
# EDA found one suspicious near-duplicate profile after exact duplicate removal.
# The rows differ only in age and fnlwgt, so we keep the first surviving row.
profile_cols = [col for col in df.columns if col not in ["age", "fnlwgt"]]
near_duplicate_indices_after_exact_dedup = [
    idx for idx in near_duplicate_indices if idx in df_no_exact_duplicates.index
]
near_duplicate_rows_after_exact_dedup = df_no_exact_duplicates.loc[
    near_duplicate_indices_after_exact_dedup
]

near_duplicate_indices_to_drop = near_duplicate_rows_after_exact_dedup[
    near_duplicate_rows_after_exact_dedup.duplicated(subset=profile_cols, keep="first")
].index

df_clean = df_no_exact_duplicates.drop(index=near_duplicate_indices_to_drop).reset_index(drop=True)

print(f"Rows before removing near-duplicates: {len(df_no_exact_duplicates)}")
print(f"Near-duplicate rows removed: {len(near_duplicate_indices_to_drop)}")
print(f"Rows after duplicate cleaning: {len(df_clean)}")

## Missing-Value Indicators

In [ ]:
missing_value_cols = ["workclass", "occupation", "native_country"]

for col in missing_value_cols:
    df_clean[f"{col}_missing"] = df_clean[col].isna().astype(int)

df_clean[[*missing_value_cols, *(f"{col}_missing" for col in missing_value_cols)]].head()

## Feature Cleaning

In [ ]:
# EDA showed education and education_num encode the same information.
# EDA also recommended dropping fnlwgt because it is a census weight.
df_model = df_clean.drop(columns=["education", "fnlwgt"])

# Capital gain/loss are highly right-skewed, so use log1p-transformed versions.
df_model["capital_gain_log"] = np.log1p(df_model["capital_gain"])
df_model["capital_loss_log"] = np.log1p(df_model["capital_loss"])
df_model = df_model.drop(columns=["capital_gain", "capital_loss"])

print(f"Modeling shape before encoding: {df_model.shape}")
df_model.head()

## Train-Test Split

In [ ]:
X = df_model.drop(columns=TARGET)
y = (df_model[TARGET] == ">50K").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True).rename({0: "<=50K", 1: ">50K"}))

## Preprocessing Pipeline

In [ ]:
class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    def __init__(self, min_frequency=0.01, other_label="Other"):
        self.min_frequency = min_frequency
        self.other_label = other_label

    def fit(self, X, y=None):
        values = pd.Series(np.asarray(X).ravel())
        frequencies = values.value_counts(normalize=True)
        self.frequent_categories_ = set(
            frequencies[frequencies >= self.min_frequency].index
        )
        return self

    def transform(self, X):
        X_array = np.asarray(X, dtype=object).copy()
        flat_values = pd.Series(X_array.ravel())
        rare_mask = ~flat_values.isin(self.frequent_categories_).to_numpy()
        X_array.ravel()[rare_mask] = self.other_label
        return X_array

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)


def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

In [ ]:
numeric_features = [
    "age",
    "education_num",
    "hours_per_week",
    "capital_gain_log",
    "capital_loss_log",
    "workclass_missing",
    "occupation_missing",
    "native_country_missing",
]

categorical_features = [
    "workclass",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
]

native_country_feature = ["native_country"]

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_one_hot_encoder()),
    ]
)

native_country_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("rare_grouper", RareCategoryGrouper(min_frequency=0.01)),
        ("onehot", make_one_hot_encoder()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
        ("country", native_country_pipeline, native_country_feature),
    ],
    remainder="drop",
)

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index,
)
X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index,
)

print(f"Processed train shape: {X_train_processed.shape}")
print(f"Processed test shape: {X_test_processed.shape}")
X_train_processed.head()

## Optional Save

In [ ]:
SAVE_OUTPUTS = False

if SAVE_OUTPUTS:
    df_clean.to_csv("adult_clean.csv", index=False)
    X_train_processed.to_csv("adult_X_train_processed.csv", index=False)
    X_test_processed.to_csv("adult_X_test_processed.csv", index=False)
    y_train.to_csv("adult_y_train.csv", index=False)
    y_test.to_csv("adult_y_test.csv", index=False)
    print("Saved cleaned and processed files.")
else:
    print("SAVE_OUTPUTS is False, so no files were written.")